# Data Query Helpers

In [1]:
import kagglehub
import math
import os
import pandas as pd
import numpy as np
import praw
import spacy
import yake
from tqdm.notebook import tqdm
from itertools import chain
from sklearn.metrics import ndcg_score
from prettytable import PrettyTable
from typing import List
from sklearn.preprocessing import MinMaxScaler
from bpemb import BPEmb
from worker import parallel_bagging

In [2]:
def download_dataset()->list[str]:
    """
    Download the dataset from Kaggle and return the paths to the files.
    """
    dataset_dir = kagglehub.dataset_download("josephleake/huge-collection-of-reddit-votes")
    paths = []
    for dir_path, _, file_names in os.walk(dataset_dir):
        for file_name in file_names:
            paths.append(os.path.join(dir_path, file_name))
    print(f'File path to votes:\n{paths[0]}')
    print(f'File path to submissions:\n{paths[1]}')
    return paths

def get_dataframe()->tuple[pd.DataFrame, pd.DataFrame]:
    """
    Return a tuple of two pandas.Dataframe: votes and submissions.

    Returns:
        tuple[pd.DataFrame]: a tuple of two dataframes.
    """
    paths = download_dataset()
    votes = pd.read_csv(paths[0], sep='\t')
    submissions = pd.read_csv(paths[1], sep='\t')
    return (votes, submissions)

def view_users_votes(votes:pd.DataFrame):
    """
    Tally up votes for each user per subreddit.
    """
    view = (
        votes
        .groupby(['USERNAME', 'SUBREDDIT', 'VOTE'])
        .size()                         # count upvotes/downvotes in each group
        .unstack(fill_value=0)          # pivot VOTE labels into columns
        .rename(columns={
            'upvote':   'num_upvotes',
            'downvote': 'num_downvotes'
        })
        .reset_index()                  # turn USERNAME & SUBREDDIT back into columns
    )
    return view

In [3]:
def filter_subreddits(
    votes: pd.DataFrame,
    num_upvotes: int = 0,
    num_downvotes: int = 0,
    total_votes: int = 0,
    num_users: int = 0,
) -> pd.Series:
    """
    Filter a DataFrame of subreddit vote counts according to given thresholds.

    Parameters:
    - votes: DataFrame with at least ['USERNAME', 'SUBREDDIT', 'num_upvotes', 'num_downvotes'] columns.
    - num_upvotes: keep rows where num_upvotes > this value (if > 0).
    - num_downvotes: keep rows where num_downvotes > this value (if > 0).
    - total_votes: keep rows where (num_upvotes + num_downvotes) > this value (if > 0).
    - num_users: keep rows where the subreddit has more than this many unique users (if > 0).

    Returns:
    - Filtered DataFrame.
    """
    # Start with an all-True mask
    mask = pd.Series(True, index=votes.index)

    # Apply upvotes threshold
    if num_upvotes > 0:
        mask &= votes['num_upvotes'] > num_upvotes

    # Apply downvotes threshold
    if num_downvotes > 0:
        mask &= votes['num_downvotes'] > num_downvotes

    # Apply total votes threshold
    if total_votes > 0:
        mask &= (votes['num_upvotes'] + votes['num_downvotes']) > total_votes

    # Apply distinct user count per subreddit threshold
    if num_users > 0:
        # Compute number of unique users for each subreddit
        user_counts = votes.groupby('SUBREDDIT')['USERNAME'].transform('nunique')
        mask &= user_counts > num_users

    # Return a series of subreddits
    return pd.Series(votes[mask].copy()['SUBREDDIT'].unique()).str.replace(r'^r/', '', regex=True)

def get_post_count(
        subreddits:pd.Series,
        submissions:pd.DataFrame
) -> pd.DataFrame:
    """Tally the number of posts for each subreddit provided in the parameters.

    Args:
        subreddits (pd.Series): a list of names of subreddits
        submissions (pd.DataFrame): a batch of submissions/posts

    Returns:
        pd.DataFrame: a dataframe that contains the name of subreddits and number of posts in them.
    """
    posts = (
        submissions
            .groupby(['SUBREDDIT'])
            .size()
            .reset_index(name='POSTS')
            .sort_values('POSTS', ascending=False)
    )
    return posts[posts['SUBREDDIT'].isin(subreddits)]

# Reddit API Helpers

In [4]:
def download_posts(
    submissions: pd.DataFrame,
    reddit_instance: praw.Reddit,
    savefile_path: str="reddit_data.csv"
):
    """Download metadata of posts from provided submissions.

    Args:
        submissions (pd.DataFrame): A DataFrame with columns "index" and "SUBMISSION_ID".
        reddit_instance (praw.Reddit): Your reddit API instance
        savefile_path (str, optional): File to which meta data is written. Defaults to "reddit_data.csv".
    """
    def make_post_entry(
        index=0,
        submission_id=0,
        title="",
        selftext="",
        num_comments=0,
        num_unique_commentators=0,
        ups=0,
        upvote_ratio=0,
        author="",
        created_utc=0,
        text_only=True,
    ):
        """
        Make data entry from post for building a DataFrame
        """
        return {
            "index": index,
            "submission_id": submission_id,
            "title": title,
            "selftext": selftext,
            "num_comments": num_comments,
            "num_unique_commentators": num_unique_commentators,
            "ups": ups,
            "upvote_ratio": upvote_ratio,
            "author": author,
            'created_utc': created_utc,
            'text_only': text_only,
        }

    # Define the schema we expect
    expected_cols = list(make_post_entry())

    try:
        # If the file exists, verify its header matches exactly
        if os.path.isfile(savefile_path):
            existing_cols = pd.read_csv(savefile_path, nrows=0).columns.tolist()
            if existing_cols != expected_cols:
                print(
                    f"Header mismatch:\n"
                    f"  existing file has columns {existing_cols}\n"
                    f"  expected columns {expected_cols}"
                )
                return None
        else:
            pd.DataFrame(columns=expected_cols).to_csv(savefile_path, index=False)
    except Exception as e:
        print(f'Invalid path {savefile_path}: {e}')

    buffer = []
    id_pattern = r"^t3_" # used for sanitizing id string
    chunk_size=100
    chunks = math.ceil(submissions.shape[0] / chunk_size)
    counter = 0

    for i in tqdm(range(chunks), "working hard at scraping", chunks):
        start = i*chunk_size
        end = (i+1)*chunk_size
        slc = slice(start, end)
        try:
            posts = reddit_instance.info(fullnames=submissions.iloc[slc]['SUBMISSION_ID'].to_list())
            for idx, post in enumerate(posts):
                entry = make_post_entry(
                    index = idx + start,
                    submission_id = post.name,
                    title = post.title,
                    selftext = post.selftext,
                    num_comments = len(post._comments_by_id),
                    num_unique_commentators = len(set(post._comments_by_id.keys())),
                    ups = post.ups,
                    upvote_ratio = post.upvote_ratio,
                    author = post.author.name if post.author is not None else "",
                    created_utc = post.created_utc,
                    text_only = (not post.is_video) and (post.media is None),
                )
                buffer.append(entry)

            df = pd.DataFrame(buffer, columns=expected_cols)
            df.to_csv(savefile_path, mode='a', header=False, index=False)
            counter += len(buffer)
            buffer.clear()
        except Exception as e:
            print(f"Error processing batch {start}-{end}: {e}")

    print(f"Downloaded {counter} posts from {len(submissions)} ids.")


# Reddit Instance

In [5]:
# Create a reddit API instance to download metadata for each post.
reddit = praw.Reddit(
    "scrapper",
    user_agent="rs_scrapper",
)

## What are you thinking in shower?

In [ ]:
votes, submissions = get_dataframe()
showerthoughts = submissions[submissions['SUBREDDIT'].isin(['Showerthoughts'])].reset_index(drop=True)

In [ ]:
# Uncomment the following code to download the posts from Reddit
# download_posts(showerthoughts, reddit)

# Vector Negaton Model

**Motivated Theory:** By removing unwanted or unrelated word meanings, a representative word vector can then be used to retrieve contents that are tuned to the liking of a user.

**Experimental Setup:**
- **Extract keywords from each post:** Using only keywords but not the entirety of a post not only reduces the size of the input but also removing stopwords that do not contribute meaningfully to the content and emoji or other out of vocabulary words that cannot be properly handled by most open-source encoder. [Yake](https://github.com/LIAAD/yake?tab=readme-ov-file) is a lighweight keyword extractor that only relies on local features in the text and does not require any pretraining or corpus.

- **Encode keywords using Byte Pair Encoding:** Byte Pair Encoding has been proven that it improves performance for NLP tasks in general when compared to other encoding methods in [Sennrich et. al., 2016](https://aclanthology.org/P16-1162.pdf), and it is widely adopted in the NLP community.

- **Build a representative vector for users:** A user profile is built by suming up all vectors of keywords from posts that are upvoted by the user. In [Widdows, 2003](https://aclanthology.org/P03-1018.pdf), he proposes that modeling the logical statement `A1 OR A2 OR A3 ... OR An` can be as simple as a linear combination of the word vectors, which he finds out later that this is the same method proposed by [Birkhoff and von Neumann, 1936](https://link.springer.com/chapter/10.1007/978-94-010-1795-4_1) for quantum logic.

- **Subtract negative keyword vectors from user profile:** [Widdows, 2003](https://aclanthology.org/P03-1018.pdf) proposes that the word meaning `a not b` should be orthogonal to the word `b` in vector space. Following this logic, this model will first construct an orthogonal basis of the subspace represented by the negative keywords(keywords from downvoted posts); then, calculate the projection of the user profile vector onto this basis; finally, subtract this projection from the user profile vector. The resulting vector is now orthogonal to all negative keyword vectors.

- **Generate recommendations for users:** The model will compute the cosine similarity scores for each post and the user profile vector. Posts with high cosine similarity score are more likely to be recommended based on the assumption that they have contents closer to the preference of the user than those with lower score. Optionally, this model can also take into account the number of upvotes received by a post, and it will return a weighted sum of the cosine similarity score and a minmax-normalized upvotes.

**Evaluations**

From the result below, the vector negation model shows a mixed performance for users. There are two situations in which a user would down vote a post: they do not like the topic or they enjoy the topic but not the position taken by the author. The vector negation model works better when a user categorically dislike a topic and performs worse if they only enjoy certain topics but take a minority stand on the issues.(Downvote ndcg measures the ndcg for downvoted posts and it is the lower the better, i.e., posts down-voted by users are ranked near the bottm of the recommendations.)

```
+-----------------------------------------------------------------------------------------------------------+
|                                            With 0 Upvote Weight                                           |
+--------------------+---------+-----------------+---------------+----------------------+-------------------+
|     User Name      |   ndcg  | ndcg@200 w/o VN | Downvote ndcg | Downvote ndcg w/o VN | # of test samples |
+--------------------+---------+-----------------+---------------+----------------------+-------------------+
|  Adventurous_Guy   |   0.0   |     0.32564     |    0.15404    |       0.13871        |        4635       |
|       Clen23       |   0.5   |       0.0       |    0.21638    |       0.18003        |       20280       |
|     CubyChris      |   0.0   |     0.14196     |      0.0      |       0.17144        |       18393       |
|    Livelogikal     |   0.0   |       0.0       |     0.3518    |       0.31081        |        2529       |
|      Mash404       |   0.0   |       0.0       |      0.0      |       0.13107        |       19043       |
|   MingeyMackrel    | 0.20185 |     0.14582     |    0.43068    |         0.0          |        9209       |
|     Raven2002      | 0.18317 |       0.0       |     0.3508    |       0.19449        |        4402       |
|     Reeses2150     |   0.0   |     0.28906     |      0.0      |         0.0          |       47285       |
|  VerbotenPublish   | 0.19656 |     0.19656     |      0.0      |         0.0          |       13976       |
|   apoeticturtle    |   0.0   |       0.0       |    0.28146    |         0.0          |       15484       |
|     baddonkey      |   0.0   |     0.15366     |      0.0      |         0.0          |       31245       |
|    daygloviking    |   0.0   |       0.0       |    0.18667    |       0.21877        |       14821       |
| locks_are_paranoid | 0.13362 |     0.22284     |      0.0      |         0.0          |        8389       |
|      lokier01      |   0.0   |     0.19055     |      0.0      |       0.13967        |       26226       |
|  mguardian_north   | 0.24053 |     0.34643     |    0.33292    |        0.2426        |        3884       |
|    pierrekrahn     | 0.20926 |     0.32432     |      0.0      |         0.0          |        6094       |
|    spockspeare     |  0.2846 |       0.0       |      0.0      |       0.18381        |        8816       |
|     stratman42     |   0.0   |       0.0       |    0.19055    |         0.0          |       32833       |
|  uncertainusurper  |   0.0   |     0.36575     |      0.5      |       0.23576        |        6961       |
+--------------------+---------+-----------------+---------------+----------------------+-------------------+ 
```

The following table shows results when the model taks upvotes from other users into account(colloborative filtering). Even though the overall performance for the ndcg score has improved, the vector negation model still shows similar performance on the `Downvote ndcg` score as before. One possible reason is that the recommender system of Reddit favors heavily towards posts up-voted by other users, by taking into account the number of upvotes received by a post, the vector negation model is implictly tracking the recommender system of Reddit, which would also convolute the effectiveness of the vector negation process.

```
+-----------------------------------------------------------------------------------------------------------+
|                                           With 0.5 Upvote Weight                                          |
+--------------------+---------+-----------------+---------------+----------------------+-------------------+
|     User Name      |   ndcg  | ndcg@200 w/o VN | Downvote ndcg | Downvote ndcg w/o VN | # of test samples |
+--------------------+---------+-----------------+---------------+----------------------+-------------------+
|  Adventurous_Guy   | 0.39084 |     0.43467     |    0.13292    |         0.0          |        4635       |
|       Clen23       |  0.353  |     0.30095     |    0.13319    |         0.0          |       20280       |
|     CubyChris      | 0.64424 |     0.54936     |      0.0      |         0.0          |       18393       |
|    Livelogikal     |   0.0   |       0.0       |    0.36964    |       0.37477        |        2529       |
|      Mash404       | 0.27448 |     0.24814     |    0.14309    |        0.195         |       19043       |
|   MingeyMackrel    | 0.24143 |      0.292      |    0.14775    |         0.0          |        9209       |
|     Raven2002      | 0.14109 |       0.0       |    0.21565    |         0.0          |        4402       |
|     Reeses2150     |  0.4476 |     0.45527     |      0.0      |         0.0          |       47285       |
|  VerbotenPublish   |   0.0   |       0.0       |      0.0      |         0.0          |       13976       |
|   apoeticturtle    |   0.0   |       0.0       |      0.0      |       0.13967        |       15484       |
|     baddonkey      |   0.0   |       0.0       |      0.0      |         0.0          |       31245       |
|    daygloviking    |   0.0   |       0.0       |    0.17579    |         0.0          |       14821       |
| locks_are_paranoid | 0.47118 |     0.43858     |      0.0      |         0.0          |        8389       |
|      lokier01      | 0.40223 |     0.50322     |      0.0      |         0.0          |       26226       |
|  mguardian_north   | 0.30978 |     0.41908     |    0.24439    |       0.18499        |        3884       |
|    pierrekrahn     | 0.53472 |      0.5234     |      0.0      |         0.0          |        6094       |
|    spockspeare     | 0.33302 |     0.14834     |    0.16667    |        0.2662        |        8816       |
|     stratman42     |   0.0   |       0.0       |      0.0      |         0.0          |       32833       |
|  uncertainusurper  | 0.49007 |     0.56538     |    0.23736    |       0.13496        |        6961       |
+--------------------+---------+-----------------+---------------+----------------------+-------------------+
```

In [8]:
def vector_negation_recommender():
    """
    A wrapper that will run and evaluate the recommender system.
    """

    showerthoughts_data = pd.read_csv('reddit_data.csv')
    showerthoughts_votes = votes[votes['SUBREDDIT'].isin(['r/Showerthoughts'])].reset_index(drop=True)
    bpemb_en = BPEmb(lang='en', vs=50000)

    def get_embeddings(row:pd.Series, word2vec: BPEmb, column:str) -> np.ndarray:
        """
        Return the word2vec embeddings for a given submission.
        """
        return word2vec.vectors[word2vec.encode_ids(" ".join(row[column]))]

    def votes_grouped_by_user(
            left:pd.DataFrame,
            right:pd.DataFrame,
            left_idx_name:str,
            right_idx_name:str,
            user_column:str,
            merged_columns:list[str],
            sort_by:list[str],
            retained_columns:list[str],
        ):
        """Votes grouped by users, then sorted by created_utc.

        Args:
            left (pd.DataFrame): A DataFrame has columns of 'SUBMISSION_ID', 'USERNAME', 'VOTE'.
            right (pd.DataFrame): A DataFrame has columns of 'submission_id', 'created_utc', 'title', 'selftext'.

        Returns:
            pd.core.groupby.generic.DataFrameGroupBy: Each iteration call to this object will return
            the user name and a DataFrame with columns of 'SUBMISSION_ID', 'VOTE', 'created_utc', 'title'
            and 'selftext'.
        """
        votes_merged = (
            left
                .merge(
                    right[merged_columns],
                left_on=left_idx_name,
                right_on=right_idx_name,
                how='left'
                )
                .drop(right_idx_name, axis=1)
                .sort_values(sort_by)
        )

        return votes_merged.groupby(user_column)[retained_columns]

    def orthonormal_basis_svd(A, tol=1e-10):
        """
        Returns an orthonormal basis for the row-space of A using SVD.
        
        Parameters
        ----------
        A : ndarray, shape (n_rows, vector_length)
            Each row is a vector you want to span.
        tol : float
            Singular values ≤ tol are treated as zero (rank-deficient case).
    
        Returns
        -------
        B : ndarray, shape (rank, vector_length)
            Each row of B is an orthonormal basis vector.
        """
        # A = U Σ Vᵀ  ;  rows live in the span of the right-singular vectors Vᵀ
        U, S, Vt = np.linalg.svd(A, full_matrices=False)
        rank = np.sum(S > tol)
        return Vt[:rank]                 # rows of Vt are already orthonormal
    
    def compute_cosine_similarity(candidate_vector:np.ndarray, target_vector:np.ndarray, eps=1e-06):
        c_norm = np.linalg.norm(candidate_vector) + eps
        t_norm = np.linalg.norm(target_vector) + eps
        similarity = (candidate_vector.T @ target_vector) / (c_norm * t_norm)
        return similarity

    def process_test_row(row:pd.Series, word2vec:BPEmb, retrival_vector:np.ndarray, up_weight):
        """
        Returns both the word embedding and the recommendation score for each test sample.
        """
        embs = get_embeddings(row, word2vec, 'bow')
        rep = embs.sum(axis=0)
        sim = compute_cosine_similarity(rep, retrival_vector)
        score = sim * (1 - up_weight) + row['normalized_ups'] * up_weight
        return rep, score

    def get_train_test_slice(size:int, split:float=0.8):
        stop = math.floor(size * split)
        train_slice = slice(0, stop)
        test_slice = slice(stop, size)
        return train_slice, test_slice

    def run_recommender(
            user:str,
            df_user:pd.DataFrame,
            submissions:pd.DataFrame,
            top_n:int,
            word2vec:BPEmb,
            use_vector_negation:bool=True,
            up_weight:float=0,
        ):
        """Generate recommendations for a given user.

        Args:
            user (str): The user of interest.
            df_user (pd.DataFrame): A dataframe that contain the user's voting history.
            submissions (pd.DataFrame): All submissions from the subreddit showerthoughts.
            top_n (int): Number of top recommendations generated by the recommender.
            word2vec (BPEmb): A word2vec model for converting tokens to embeddings.
            use_vector_negation (bool, optional): Whether to use vector negation. Defaults to True.
            up_weight (float, optional): The weight used for total upvotes of a post.
            1 implies the model only uses colloborative filtering. Defaults to 0.

        Returns:
            _type_: _description_
        """

        submissions = submissions.sort_values('created_utc')
        bow = df_user['bow']
        up_vote_mask = df_user['VOTE'].isin(['upvote'])

        train_slice, test_slice = get_train_test_slice(up_vote_mask.shape[0])
        last_train_sample_idx = train_slice.stop - 1
        if up_vote_mask[test_slice].sum() == 0:
            print(f'No upvoted sample for user {user} in test data.')
            return (None, 0)

        pos_bow = set(chain.from_iterable(bow[train_slice][up_vote_mask[train_slice]]))
        pos_emb = word2vec.vectors[word2vec.encode_ids(" ".join(pos_bow))]
        pos_sum = pos_emb.sum(axis=0)

        # Checks if there is at least one downvote in the training set.
        orthonormal_basis  = None
        if use_vector_negation:
            if (~up_vote_mask[train_slice]).sum() > 0 :
                neg_bow = set(chain.from_iterable(bow[train_slice][~up_vote_mask[train_slice]]))

                # Removes words that are in the bag of positive words
                neg_bow_purged = neg_bow - pos_bow
                neg_emb = word2vec.vectors[word2vec.encode_ids(" ".join(neg_bow_purged))]

                # Computes the orthogonal basis for the negative embeddings.
                orthonormal_basis = orthonormal_basis_svd(neg_emb)

                # Calculates projection of pos_sum to the down_vote subspace
                proj = (orthonormal_basis @ pos_sum).T @ orthonormal_basis

                # Gets the vector that is orthogonal to the subspace of negative words
                copy = pos_sum
                pos_sum -= proj

        future_submissions_mask = submissions['created_utc'] > df_user.iloc[last_train_sample_idx]['created_utc']
        future_submissions = submissions[future_submissions_mask].copy()


        future_submissions[['embedding', 'score']] = future_submissions.apply(
            process_test_row,
            axis=1,
            word2vec = word2vec,
            retrival_vector = pos_sum,
            up_weight = up_weight,
            result_type='expand'
        )
        future_submissions.sort_values('score', inplace=True, ascending=False)
        return future_submissions[:top_n], future_submissions.shape[0]

    def parallel_bow(df:pd.DataFrame, num_processes:int=8):
        # Warning! CPU heavy function.
        # Running parallel processes to create bag of words for each submission.
        slice_size = math.ceil(df.shape[0] / num_processes)
        df_chunks = [df[i*slice_size:(i+1)*slice_size] for i in range(num_processes)]
        results = parallel_bagging(df_chunks)

        return pd.concat(results)

    bagging_results = parallel_bow(showerthoughts_data)
    showerthoughts_data['bow'] = bagging_results
    showerthoughts_data.sort_values('created_utc', inplace=True)

    scaler = MinMaxScaler()
    showerthoughts_data['normalized_ups'] = scaler.fit_transform(showerthoughts_data['ups'].to_numpy().reshape(-1,1))

    top_20_users = ['mguardian_north', 'Raven2002', 'Clen23', 'Reeses2150', 'spockspeare', 'apoeticturtle', 'Mash404', 'locks_are_paranoid', 'daygloviking', 'MingeyMackrel', 'thx1138jr', 'Livelogikal', 'baddonkey', 'uncertainusurper', 'lokier01', 'CubyChris', 'pierrekrahn', 'Adventurous_Guy', 'VerbotenPublish', 'stratman42']

    def evaluate_recommender(
            recommender,
            votes:pd.DataFrame,
            submissions:pd.DataFrame,
            users:List[str]=None,
            **kws
        ):
        """
        Prints out the metrics for the recommender.
        """
        votes_from_users = votes
        if users is not None and len(users) > 0:
            votes_from_users = votes[votes['USERNAME'].isin(users)]

        grouped = votes_grouped_by_user(
            left=votes_from_users,
            right=submissions,
            left_idx_name='SUBMISSION_ID',
            right_idx_name='submission_id',
            user_column='USERNAME',
            merged_columns=['submission_id', 'created_utc', 'title', 'selftext', 'bow'],
            sort_by=['USERNAME', 'created_utc'],
            retained_columns=['SUBMISSION_ID', 'VOTE', 'created_utc', 'title', 'selftext', 'bow'],
        )

        def get_ndcg(y_true, recommendations):
            return ndcg_score(
                np.asarray([y_true.to_list()]),
                np.asarray([recommendations['score'].to_list()])
            )

        def process_recommendations(recommendations:pd.DataFrame, df_user:pd.DataFrame):
            if recommendations is not None:
                _, test_slice = get_train_test_slice(df_user.shape[0])
                test_set = df_user[test_slice]

                upvotes = test_set[test_set['VOTE'].isin(['upvote'])]['SUBMISSION_ID']
                y_true_upvotes = recommendations['submission_id'].isin(upvotes).astype(int)

                downvotes = test_set[test_set['VOTE'].isin(['downvote'])]['SUBMISSION_ID']
                y_true_downvotes = recommendations['submission_id'].isin(downvotes).astype(int)

                return (
                    get_ndcg(y_true_upvotes, recommendations),
                    get_ndcg(y_true_downvotes, recommendations),
                )

        users = []
        ndcg = []
        table = PrettyTable(['User Name', 'ndcg', 'ndcg@200 w/o VN', 'Downvote ndcg', 'Downvote ndcg w/o VN', '# of test samples'])
        up_weight = kws.get('up_weight', 0)
        table.title = f'With {up_weight} Upvote Weight'
        for user, df_user in grouped:
            recommendations, test_size = recommender(user, df_user, submissions, **kws)
            recommendations_wo_VN, _ = recommender(user, df_user, submissions, use_vector_negation=False, **kws)
            if recommendations is None or recommendations_wo_VN is None:
                continue
            ndcg_up, ndcg_down = process_recommendations(recommendations, df_user)
            ndcg_up_wo_VN, ndcg_down_wo_VN = process_recommendations(recommendations_wo_VN, df_user)

            table.add_row([user, f'{ndcg_up:.5}', f'{ndcg_up_wo_VN:.5}', f'{ndcg_down:.5}', f'{ndcg_down_wo_VN:.5}', test_size])
            users.append(user)
            ndcg.append(f'{ndcg_up:.5}')
        print(table)
        results = pd.DataFrame({
            'USERNAME': users,
            'ndcg': ndcg,
        })
        up_weight = str(up_weight).replace('.', '')
        results.to_csv(f'vn_cf_{up_weight}.csv')

    evaluate_recommender(
        run_recommender,
        showerthoughts_votes,
        showerthoughts_data,
        users=top_20_users,
        top_n=200,
        word2vec=bpemb_en,
        up_weight=0,
    )

    evaluate_recommender(
        run_recommender,
        showerthoughts_votes,
        showerthoughts_data,
        users=top_20_users,
        top_n=200,
        word2vec=bpemb_en,
        up_weight=0.25,
    )

    evaluate_recommender(
        run_recommender,
        showerthoughts_votes,
        showerthoughts_data,
        users=top_20_users,
        top_n=200,
        word2vec=bpemb_en,
        up_weight=0.5,
    )

    evaluate_recommender(
        run_recommender,
        showerthoughts_votes,
        showerthoughts_data,
        users=top_20_users,
        top_n=200,
        word2vec=bpemb_en,
        up_weight=0.75,
    )

    evaluate_recommender(
        run_recommender,
        showerthoughts_votes,
        showerthoughts_data,
        users=top_20_users,
        top_n=200,
        word2vec=bpemb_en,
        up_weight=1,
    )

In [9]:
vector_negation_recommender()

No upvoted sample for user thx1138jr in test data.
No upvoted sample for user thx1138jr in test data.
+-----------------------------------------------------------------------------------------------------------+
|                                            With 0 Upvote Weight                                           |
+--------------------+---------+-----------------+---------------+----------------------+-------------------+
|     User Name      |   ndcg  | ndcg@200 w/o VN | Downvote ndcg | Downvote ndcg w/o VN | # of test samples |
+--------------------+---------+-----------------+---------------+----------------------+-------------------+
|  Adventurous_Guy   |   0.0   |     0.32564     |    0.15404    |       0.13871        |        4635       |
|       Clen23       |   0.5   |       0.0       |    0.21638    |       0.18003        |       20280       |
|     CubyChris      |   0.0   |     0.14196     |      0.0      |       0.17144        |       18393       |
|    Livelogikal  